## Delegating Work with Handoffs

# Agentic Patterns: The Handoff Pattern

## Introduction & Overview

Welcome to another lesson about agentic patterns! In the previous lesson, you mastered orchestrating agents as tools, where a central planner agent could dynamically delegate tasks to specialist agents and receive their results back. Today, we're exploring a fundamentally different approach called the **handoff pattern**, where agents can completely transfer control to other specialized agents rather than just calling them as tools.

In this lesson, you'll extend the `Agent` class constructor to support handoff targets, create a handoff tool schema that enables control transfers, and implement the core handoff logic that cleanly passes conversation context between agents. We'll build a practical example with a general assistant that can hand off mathematical problems to a specialized calculator assistant, demonstrating how agents make intelligent decisions about when to transfer control versus handling tasks themselves.

## Understanding the Handoff Pattern

The handoff pattern represents a different philosophy of agent collaboration compared to the tool delegation approach you learned previously.

* **Tool Delegation:** The agent asks for help while maintaining responsibility for the final response. The user interacts only with the orchestrator.
* **Handoff Pattern:** The original agent says, "this other agent is better equipped to handle this entire conversation from here on." The specialist agent then responds directly to the user, and the original agent is no longer involved.

This pattern is particularly powerful when you have agents with very different capabilities or when the nature of a request clearly falls into one agent's domain of expertise.

## Extending the Agent Class Constructor

To implement handoffs, we need to extend our existing `Agent` class with the ability to track available handoff targets.

```python
class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,  # New parameter for handoff targets
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns
        
        # Avoid shared mutable defaults
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        self.handoffs = [] if handoffs is None else list(handoffs)  # List of agents for handoffs

```

## Creating the Handoff Tool Schema

We need a tool schema that allows the agent to request handoffs. This schema is automatically added to the agent's available tools when handoff targets are provided.

```python
# Define handoff tool schema
self.handoff_schema = {
    "name": "handoff",
    "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
            },
            "reason": {
                "type": "string", 
                "description": "Brief explanation of why this handoff is needed"
            }
        },
        "required": ["name", "reason"]
    }
}

```

## Implementing the Handoff Logic

The `call_handoff` method handles the transfer of control. It performs several critical steps:

1. **Parameter Extraction:** Gets the target name and reason.
2. **Agent Lookup:** Finds the target agent in the `handoffs` list.
3. **Context Cleaning:** Removes the assistant's message containing the handoff call so the target agent receives a clean history.
4. **Control Transfer:** Calls the target agent's `run` method.

## When to Use Agents as Tools vs Handoffs

* **Use Agents as Tools:** When you need an orchestrator to maintain control and synthesize multiple inputs into a unified response (e.g., planning a trip).
* **Use Handoffs:** When a specialist is better equipped to handle the *entire* conversation from a certain point forward (e.g., passing a general query to a dedicated math or coding specialist).

## Summary & Best Practices

* **Define Clear Criteria:** Use prompts to help agents decide exactly when to transfer.
* **Context Management:** Always clean the conversation history before the handoff.
* **Robust Error Handling:** Ensure your system can recover if a handoff target is not found or fails.
* **Avoid Circularity:** Design handoff chains with clear directionality to prevent agents from passing control back and forth indefinitely.

## Building the Handoff Infrastructure

It's time to build the foundation that will enable agents to transfer control to one another! In this exercise, you'll extend the Agent class to support handoffs by adding some of the necessary infrastructure.

    Modify the Agent class constructor in src/agent.py:
        Add a handoffs parameter (default to None)
        Initialize self.handoffs as an empty list if None, otherwise copy the provided list

    Create the self.handoff_schema dictionary that defines the handoff tool:
        Set tool name to "handoff"
        Add a description explaining it transfers control to specialized agents
        Define input_schema with two required properties:
            "name": target agent's name
            "reason": why the handoff is needed
        Use a list comprehension over self.handoffs to dynamically list available agents in the "name" property's description

    Update src/main.py:
        Add handoffs=[calculator_assistant] to the helpful_assistant agent
        Print the handoff_schema attribute to verify the implementation

The displayed schema will show exactly how your agent will see the handoff tool when deciding whether to use it.

```
# agent.py
import json
import anthropic


class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    # TODO: Add the handoffs parameter to the constructor (default to None)
    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        # TODO: Initialize self.handoffs as an empty list if None, otherwise create a copy of the provided list
        
        # TODO: Define self.handoff_schema as a dictionary with the handoff tool schema
        # The schema should have:
        # - "name": "handoff"
        # - "description": explaining it transfers control to specialized agents
        # - "input_schema" with "name" and "reason" properties (both required)
        # - The "name" property description should dynamically list available agents from self.handoffs
        
    def _extract_text(self, content):
        return "".join(
            block.text for block in content
            if getattr(block, "type", None) == "text"
        )    

    def _build_request_args(self, messages):
        request_args = {
            "model": self.model,
            "system": self.system_prompt,
            "messages": messages,
            "max_tokens": 8000,
        }

        # Add tool schemas only if they exist
        if self.tool_schemas:
            request_args["tools"] = self.tool_schemas

        return request_args

    def call_tool(self, tool_use):
        tool_name = tool_use.name
        tool_input = tool_use.input or {}
        tool_use_id = tool_use.id

        print(f"🔧 Tool called: {tool_name}({tool_input})")
        
        try:
            result = str(self.tools[tool_name](**tool_input))
        except KeyError:
            result = f"Error: Tool {tool_name} not found"
        except Exception as e:
            result = f"Error: {str(e)}"

        return {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result
        }

    def run(self, input_messages):
        # Create a copy of the input messages to avoid modifying the original
        messages = input_messages.copy()

        # Loop until the model returns a final answer or the max turns is reached
        turn = 0
        while turn < self.max_turns:
            turn += 1

            response = self.client.messages.create(**self._build_request_args(messages))

            messages.append({"role": "assistant", "content": response.content})

            # Execute all tools if Claude requests any
            if response.stop_reason == "tool_use":
                tool_results = []
                for content_item in response.content:
                    if content_item.type == "tool_use":
                        tool_result = self.call_tool(content_item)
                        tool_results.append(tool_result)

                # Add all tool results to messages
                messages.append({
                    "role": "user",
                    "content": tool_results
                })
        
            else:
                # Return if no tools are requested
                response_text = self._extract_text(response.content)

                return messages, response_text

        # If the max turns is reached, raise an exception
        raise Exception("Max turns reached")


# main.py

import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# TODO: Add the handoffs parameter to helpful_assistant, setting it to [calculator_assistant]
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems."
)

# TODO: Print the handoff_schema attribute from helpful_assistant to verify it was created
# Hint: Use json.dumps with indent=2 for pretty printing
```

Here is the updated code for `agent.py` and `main.py` with the handoff infrastructure implemented as requested.

### 1. Updated `agent.py`

I have added the `handoffs` parameter to the `__init__` method, initialized the handoff list, and created the `self.handoff_schema` definition.

```python
import json
import anthropic

class Agent:
    BASE_SYSTEM_PROMPT = (
        "You are an autonomous agent that can take multiple tool-calling steps when helpful. "
        "The user only sees your response when you stop using tools, not your tool usage or reasoning steps. "
        "When you provide your answer without calling tools, make it complete and standalone.\n"
        "Additional instructions:\n"
    )

    def __init__(
        self,
        name,
        system_prompt="You are a helpful assistant.",
        model="claude-sonnet-4-6",
        tools=None,
        tool_schemas=None,
        handoffs=None,  # Added handoffs parameter
        max_turns=10
    ):
        self.client = anthropic.Anthropic()
        self.name = name
        self.model = model
        self.system_prompt = self.BASE_SYSTEM_PROMPT + system_prompt
        self.max_turns = max_turns

        # Avoid shared mutable defaults and protect against external mutation
        self.tools = {} if tools is None else dict(tools)
        self.tool_schemas = [] if tool_schemas is None else list(tool_schemas)
        
        # Initialize handoffs list
        self.handoffs = [] if handoffs is None else list(handoffs)

        # Define handoff tool schema
        self.handoff_schema = {
            "name": "handoff",
            "description": "Transfer control to another specialized agent. Use this when the user's request is better handled by a different agent.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": f"Name of the agent to handoff to. Available agents: {[agent.name for agent in self.handoffs]}"
                    },
                    "reason": {
                        "type": "string",
                        "description": "Brief explanation of why this handoff is needed"
                    }
                },
                "required": ["name", "reason"]
            }
        }

    # ... (rest of the Agent class methods remain the same)

```

---

### 2. Updated `main.py`

I have updated the `helpful_assistant` instantiation to include the `handoffs` parameter and added the print statement to verify the generated schema.

```python
import json
from agent import Agent
from functions import (
    sum_numbers,
    multiply_numbers,
    subtract_numbers,
    divide_numbers,
    power,
    square_root
)

# Load tool schemas
with open('schemas.json', 'r') as f:
    tool_schemas = json.load(f)

# Math tools
math_tools = {
    "sum_numbers": sum_numbers,
    "multiply_numbers": multiply_numbers,
    "subtract_numbers": subtract_numbers,
    "divide_numbers": divide_numbers,
    "power": power,
    "square_root": square_root
}

# Create a calculator assistant
calculator_assistant = Agent(
    name="calculator_assistant",
    system_prompt="You are a calculator assistant. You specialize in mathematical calculations and solving equations.",
    tools=math_tools,
    tool_schemas=tool_schemas
)

# Add handoffs to helpful_assistant
helpful_assistant = Agent(
    name="helpful_assistant",
    system_prompt="You are a helpful assistant. You can assist with various tasks and handoff to the calculator assistant for math problems.",
    handoffs=[calculator_assistant]
)

# Print the handoff_schema attribute to verify
print("=== Handoff Schema ===")
print(json.dumps(helpful_assistant.handoff_schema, indent=2))

```

When you run this updated `main.py`, you will see a JSON printout of the `handoff` tool schema, confirming that the list of available agents (`['calculator_assistant']`) has been dynamically injected into the tool description.

## Combining Tool Schemas for Agent Requests

## Implementing the Core Handoff Logic

## Detecting Handoffs and Transferring Control